In [1]:
values = [1, 2, 4, 2, 4, 5, 2, 4]


In [2]:
mid = len(values) // 2
part1 = values[:mid]
part2 = values[mid:]


In [3]:
import numpy as np

def pct_change(data):
    data = np.array(data, dtype=float)
    return (data[1:] - data[:-1]) / data[:-1] * 100


In [4]:
pct1 = pct_change(part1)
pct2 = pct_change(part2)

std1 = np.std(pct1, ddof=1)  # sample std
std2 = np.std(pct2, ddof=1)

print("Part 1 % changes:", pct1)
print("Part 1 std dev:", std1)

print("Part 2 % changes:", pct2)
print("Part 2 std dev:", std2)


Part 1 % changes: [100. 100. -50.]
Part 1 std dev: 86.60254037844386
Part 2 % changes: [ 25. -60. 100.]
Part 2 std dev: 80.05206639015219


In [5]:
import numpy as np


In [6]:
values = np.array([1, 2, 4, 2, 4, 5, 2, 4], dtype=float)


In [7]:
def pct_change(arr):
    return (arr[1:] - arr[:-1]) / arr[:-1] * 100


In [8]:
def rolling_volatility_split(data, window):
    """
    Rolling window volatility comparison.
    Splits each window into 2 halves.
    Returns:
        std_first, std_second, volatility_ratio
    """
    data = np.asarray(data, dtype=float)
    n = len(data)
    half = window // 2

    std_first = []
    std_second = []
    ratio = []

    for i in range(window, n + 1):
        window_data = data[i - window:i]

        first = window_data[:half]
        second = window_data[half:]

        pct1 = pct_change(first)
        pct2 = pct_change(second)

        s1 = np.std(pct1, ddof=1) if len(pct1) > 1 else 0
        s2 = np.std(pct2, ddof=1) if len(pct2) > 1 else 0

        std_first.append(s1)
        std_second.append(s2)
        ratio.append(s2 / s1 if s1 != 0 else np.nan)

    return np.array(std_first), np.array(std_second), np.array(ratio)


In [9]:
std1, std2, vol_ratio = rolling_volatility_split(values, window=6)

print("First half volatility :", std1)
print("Second half volatility:", std2)
print("Volatility ratio      :", vol_ratio)


First half volatility : [  0.         106.06601718 106.06601718]
Second half volatility: [ 53.03300859  60.1040764  113.13708499]
Volatility ratio      : [       nan 0.56666667 1.06666667]


In [10]:
def volatility_features(data, window):
    std1, std2, ratio = rolling_volatility_split(data, window)

    return {
        "vol_first_half": std1,
        "vol_second_half": std2,
        "vol_ratio": ratio
    }


In [15]:
features = volatility_features(values, window=6)
print(f'Vol ratio: {features["vol_ratio"][-1]}')
print(f'Vol first half: {features["vol_first_half"][-1]}')
print(f'Vol second half: {features["vol_second_half"][-1]}')


Vol ratio: 1.0666666666666667
Vol first half: 106.06601717798213
Vol second half: 113.13708498984761


In [16]:
import numpy as np
import pandas as pd


In [17]:
def pct_change(arr):
    arr = np.asarray(arr, dtype=float)
    return (arr[1:] - arr[:-1]) / arr[:-1] * 100


In [18]:
def rolling_volatility_split(data, window):
    data = np.asarray(data, dtype=float)
    half = window // 2

    v1, v2, ratio = [], [], []

    for i in range(window, len(data) + 1):
        w = data[i - window:i]

        f, s = w[:half], w[half:]

        p1, p2 = pct_change(f), pct_change(s)

        std1 = np.std(p1, ddof=1) if len(p1) > 1 else 0
        std2 = np.std(p2, ddof=1) if len(p2) > 1 else 0

        v1.append(std1)
        v2.append(std2)
        ratio.append(std2 / std1 if std1 != 0 else np.nan)

    return np.array(v1), np.array(v2), np.array(ratio)


In [19]:
def zscore(arr, window=20):
    s = pd.Series(arr)
    return ((s - s.rolling(window).mean()) /
            s.rolling(window).std()).values


In [21]:
def zscore(arr, window=20):
    s = pd.Series(arr)
    return ((s - s.rolling(window).mean()) /
            s.rolling(window).std()).values


In [22]:
def atr(high, low, close, window=14):
    high, low, close = map(pd.Series, (high, low, close))
    tr = pd.concat([
        high - low,
        (high - close.shift()).abs(),
        (low - close.shift()).abs()
    ], axis=1).max(axis=1)

    return tr.rolling(window).mean().values


In [23]:
def classify_regime(vol_ratio):
    if np.isnan(vol_ratio):
        return "UNDEFINED"
    elif vol_ratio < 0.7:
        return "COMPRESSION"
    elif vol_ratio < 1.0:
        return "STABLE"
    elif vol_ratio < 1.5:
        return "EXPANSION"
    else:
        return "VOLATILITY_SPIKE"


In [24]:
def trade_signal(regime):
    if regime == "EXPANSION":
        return 1      # ENTER
    elif regime == "COMPRESSION":
        return -1     # AVOID / EXIT
    else:
        return 0      # HOLD


In [25]:
def build_volatility_features(df, price_col="close", window=20):
    prices = df[price_col].values

    v1, v2, ratio = rolling_volatility_split(prices, window)

    pad = len(df) - len(ratio)

    df_feat = df.iloc[pad:].copy()
    df_feat["vol_first"] = v1
    df_feat["vol_second"] = v2
    df_feat["vol_ratio"] = ratio
    df_feat["vol_ratio_z"] = zscore(ratio)

    df_feat["regime"] = df_feat["vol_ratio"].apply(classify_regime)
    df_feat["signal"] = df_feat["regime"].apply(trade_signal)

    return df_feat


| Column        | Meaning                 |
| ------------- | ----------------------- |
| `vol_first`   | Early window volatility |
| `vol_second`  | Recent volatility       |
| `vol_ratio`   | Regime strength         |
| `vol_ratio_z` | Normalized regime       |
| `regime`      | Market state            |
| `signal`      | Trading action          |


| Signal | Meaning           |
| ------ | ----------------- |
| `1`    | Enter trend trade |
| `0`    | Hold / wait       |
| `-1`   | Avoid / exit      |


In [ ]:
# Example price data
df = pd.DataFrame({
    "close": [1,2,4,2,4,5,2,4,6,7,5,8,9,10,11,12,13,14,15,16,17,18,19,20]
})

features = build_volatility_features(df, window=6)
features


,close,vol_first,vol_second,vol_ratio,vol_ratio_z,regime,signal
5,5,0.000000,53.033009,NaN,NaN,UNDEFINED,0
6,2,106.066017,60.104076,0.566667,NaN,COMPRESSION,-1
7,4,106.066017,113.137085,1.066667,NaN,EXPANSION,1
8,6,53.033009,35.355339,0.666667,NaN,COMPRESSION,-1
9,7,60.104076,23.570226,0.392157,NaN,COMPRESSION,-1
10,5,113.137085,31.988164,0.282738,NaN,COMPRESSION,-1
11,8,35.355339,62.629458,1.771429,NaN,VOLATILITY_SPIKE,0
12,9,23.570226,33.587572,1.425000,NaN,EXPANSION,1
13,10,31.988164,0.982093,0.030702,NaN,COMPRESSION,-1
14,11,62.629458,0.785674,0.012545,NaN,COMPRESSION,-1


🧠 ML / RL INTEGRATION
ML features
X = features[["vol_ratio", "vol_ratio_z"]]
y = features["signal"]

RL state vector
state = np.array([
    features["vol_ratio"].iloc[-1],
    features["vol_ratio_z"].iloc[-1]
])

🚀 WHY THIS IS POWERFUL

✔ Detects regime shifts early
✔ Avoids chop & fake breakouts
✔ Works on 5-sec candles
✔ Lightweight & fast
✔ ML & RL friendly

In [ ]:
import pandas as pd
import numpy as np

class VolatilityIndicator(BaseIndicator):
    def __init__(self, config: Dict[str, Any] = None):
        super().__init__(config)
        self.window = config.get('window', 20)
        
    def calculate(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        prices = df['close'].values
        v1, v2, ratio = rolling_volatility_split(prices, self.window)
        
        pad = len(df) - len(ratio)
        df = df.iloc[pad:].copy()
        df['vol_first'] = v1
        df['vol_second'] = v2
        df['vol_ratio'] = ratio
        df['vol_ratio_z'] = zscore(ratio)
        
        # Generate signals based on volatility regime
        df['signal'] = 'HOLD'
        df.loc[df['vol_ratio_z'] > 1, 'signal'] = 'BUY'  # High volatility, trend might change
        df.loc[df['vol_ratio_z'] < -1, 'signal'] = 'SELL'  # Low volatility, trend might continue
        
        # Trend detection
        df['trend'] = np.where(df['close'] > df['close'].shift(self.window), 'UP', 'DOWN')
        df.loc[df['close'] == df['close'].shift(self.window), 'trend'] = 'NEUTRAL'
        
        # Adjust signals based on trend
        df.loc[(df['signal'] == 'BUY') & (df['trend'] == 'DOWN'), 'signal'] = 'HOLD'
        df.loc[(df['signal'] == 'SELL') & (df['trend'] == 'UP'), 'signal'] = 'HOLD'
        
        return df
    
    def get_required_columns(self) -> list:
        return ['close']

In [ ]:
indicator = VolatilityIndicator({'window': 6})
df = pd.DataFrame({
    "close": [1,2,4,2,4,5,2,4,6,7,5,8,9,10,11,12,13,14,15,16,17,18,19,20]
})
df = indicator.calculate(df)
print(df)

In [26]:
from Data.fyers_data_final import FyersDataScanner
scanner = FyersDataScanner()
start_date = '2026-02-01 09:15:00'
end_date = '2026-02-02 16:15:00'
# - BSE:SENSEX2620581000PE
# - BSE:SENSEX2620582200CE
symbol = 'BSE:SENSEX2620582200CE'
# resolution = "5S"

resolutions = ["1", "5S", "15S", "30S"]
for resolution in resolutions:
    # Fetch all data once at initialization
    scanner.start_date = start_date
    scanner.end_date = end_date
    scanner.symbol = symbol
    scanner.resolution = resolution

    print(f"📥 Fetching all data for {symbol}...")
    full_data = scanner.get_data()
    full_data.to_csv(f'{symbol}_{resolution}.csv')

2026-02-03 01:57:06.898 | INFO     | Data.fyers_data_final:get_data:21 - Getting data for BSE:SENSEX2620582200CE from 2026-02-01 09:15:00 to 2026-02-02 16:15:00 
2026-02-03 01:57:06.901 | INFO     | Data.fyers_data_final:get_data:22 - Resolution: 1
2026-02-03 01:57:06.902 | INFO     | Data.DATA_SOURCE:get_data:66 - Cache name: fyers_cache_BSE:SENSEX2620582200CE_1769937300_1770048900_1


📥 Fetching all data for BSE:SENSEX2620582200CE...


/home/sham/Desktop/TradeMagic/new/AgenticTrading/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api-t1.fyers.in'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
2026-02-03 01:57:07.169 | ERROR    | Data.DATA_SOURCE:get_data:101 - Error in fetching data: 6 columns passed, passed data had 7 columns
2026-02-03 01:57:07.213 | INFO     | Data.fyers_data_final:get_data:36 - Data: 795 rows
2026-02-03 01:57:07.235 | INFO     | Data.fyers_data_final:get_data:21 - Getting data for BSE:SENSEX2620582200CE from 2026-02-01 09:15:00 to 2026-02-02 16:15:00 
2026-02-03 01:57:07.236 | INFO     | Data.fyers_data_final:get_data:22 - Resolution: 5S
2026-02-03 01:57:07.237 | INFO     | Data.DATA_SOURCE:get_data:66 - Cache name: fyers_cache_BSE:SENSEX2620582200CE_1769937300_1770048900_5S
/home/sham/Desktop/TradeMagic/

📥 Fetching all data for BSE:SENSEX2620582200CE...


2026-02-03 01:57:07.703 | ERROR    | Data.DATA_SOURCE:get_data:101 - Error in fetching data: 6 columns passed, passed data had 7 columns
2026-02-03 01:57:07.948 | INFO     | Data.fyers_data_final:get_data:36 - Data: 9540 rows
2026-02-03 01:57:08.053 | INFO     | Data.fyers_data_final:get_data:21 - Getting data for BSE:SENSEX2620582200CE from 2026-02-01 09:15:00 to 2026-02-02 16:15:00 
2026-02-03 01:57:08.054 | INFO     | Data.fyers_data_final:get_data:22 - Resolution: 15S
2026-02-03 01:57:08.055 | INFO     | Data.DATA_SOURCE:get_data:66 - Cache name: fyers_cache_BSE:SENSEX2620582200CE_1769937300_1770048900_15S
/home/sham/Desktop/TradeMagic/new/AgenticTrading/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api-t1.fyers.in'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


📥 Fetching all data for BSE:SENSEX2620582200CE...


2026-02-03 01:57:08.317 | ERROR    | Data.DATA_SOURCE:get_data:101 - Error in fetching data: 6 columns passed, passed data had 7 columns
2026-02-03 01:57:08.494 | INFO     | Data.fyers_data_final:get_data:36 - Data: 3180 rows
2026-02-03 01:57:08.538 | INFO     | Data.fyers_data_final:get_data:21 - Getting data for BSE:SENSEX2620582200CE from 2026-02-01 09:15:00 to 2026-02-02 16:15:00 
2026-02-03 01:57:08.539 | INFO     | Data.fyers_data_final:get_data:22 - Resolution: 30S
2026-02-03 01:57:08.540 | INFO     | Data.DATA_SOURCE:get_data:66 - Cache name: fyers_cache_BSE:SENSEX2620582200CE_1769937300_1770048900_30S
/home/sham/Desktop/TradeMagic/new/AgenticTrading/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api-t1.fyers.in'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


📥 Fetching all data for BSE:SENSEX2620582200CE...


2026-02-03 01:57:08.799 | ERROR    | Data.DATA_SOURCE:get_data:101 - Error in fetching data: 6 columns passed, passed data had 7 columns
2026-02-03 01:57:08.854 | INFO     | Data.fyers_data_final:get_data:36 - Data: 1590 rows


In [29]:
from trading_system.core.indicator_manager import IndicatorManager
import pandas as pd

In [30]:
df = pd.read_csv('/home/sham/Desktop/TradeMagic/new/AgenticTrading/BSE:SENSEX2620582200CE_30S.csv')

In [38]:
indicator_manager = IndicatorManager()

In [39]:
# v = d.execute_all(df)

In [41]:
# Execute all indicators
indicator_signals = indicator_manager.execute_all(df, verbose=True)

# Get indicator weights
weights = indicator_manager.get_weights()



🔍 Executing 0 indicators...


✅ Execution complete!

